In [1]:
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display, Audio
import lightning as L
import sys
from lightning.pytorch.callbacks import LearningRateMonitor
from lightning_scripts.lightning_ssl_matched_speech_in_noise import LitAudioSSL
from lightning_scripts.jsinV3DataLoader_precombined_batched import CleanSpeechInNoiseValDatasetBatched

sys.path.append('../')
import importlib
import yaml
import torch
import os 
from pathlib import Path
import pickle
from lightning_scripts.eval_jsin_transfer_matched import SSLClassifier
from tqdm.auto import tqdm

/mnt/ceph/users/igriffith/projects/cochdnn/byol-a/byol_a/common.py:31: UserWarning: torchaudio._backend.set_audio_backend has been deprecated. With dispatcher enabled, this function is no-op. You can remove the function call.
  torchaudio.set_audio_backend("sox_io")


In [2]:
## init config. Will be yaml eventually, but start as dict 
config_path = Path("model_configs/resnet50_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01.yaml")
config = yaml.load(open(config_path, 'r'), Loader=yaml.FullLoader)

config['num_workers'] = 4
config['hparas']['batch_size'] = 64
config['data']['eval_max'] = 3

exp_dir = Path("model_checkpoints")
checkpoint_dir = exp_dir / f"{config_path.stem}/checkpoints"

ckpt_paths = sorted(checkpoint_dir.glob("*.ckpt"), key=os.path.getctime)

ckpt_path = ckpt_paths[-1] 
module = LitAudioSSL.load_from_checkpoint(checkpoint_path=ckpt_path, config=config).eval().cuda()

In [3]:
ckpt_path

PosixPath('model_checkpoints/resnet50_barlow_equivariant_lmbda_1e-2_lr_2e-1_eq_lmbda_5e-01/checkpoints/epoch=106-step=48150-best_train.ckpt')

In [4]:
# run test 
import robustness.audio_functions.audio_transforms as at
transforms = at.AudioCompose([
                at.AudioToTensor(),
                at.CombineWithRandomDBSNR(low_snr=6,
                                        high_snr=6),
                at.DBSPLNormalizeForegroundAndBackground(dbspl=60),
                at.UnsqueezeAudio(dim=0) # dim=0 here so batches of audio from dataloader will be (Batch, 1, Time)
            ])

def collate_fn(batch):
    audio, noise, targets = batch[0] # unbox wrapper added by dataloader 
    audio = audio.unsqueeze(1)
    # # combine labels: each target is dict for each key, stack the values 
    output_audio = []
    labels = {}
    for label_key in targets.keys():
        labels[label_key] = torch.from_numpy(targets[label_key])

    for aud_eg, noise_eg in zip(audio, noise):
        aud_eg, _  = transforms(aud_eg.squeeze().numpy(), noise_eg.numpy())
        output_audio.append(aud_eg)
    output_audio = torch.stack(output_audio, dim=0)
    return output_audio, labels

test_dataset = CleanSpeechInNoiseValDatasetBatched(config['data']['speech_h5_path'],
                                            target_keys=config['data']['target_keys'],
                                            return_noise=True,
                                            batch_size=100
)
test_dataset.target_keys = ['signal/word_int']
test_dataloader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=1,
    num_workers=config['num_workers'],
    shuffle=False,
    collate_fn=collate_fn
)


audio, labels = next(iter(test_dataloader))

In [5]:
audio, labels = next(iter(test_dataloader))

In [6]:

n_batches = 500

top1_word = []
top5_word = []

for ix, (audio, labels) in enumerate(tqdm(test_dataloader, total=n_batches)):
    if ix == n_batches:
        break
    feature, out, logits = module(audio.cuda())
    word_logits = logits['signal/word_int'].cpu()
    word_labels = labels['signal/word_int']
    # task_IXS = (word_labels != 0 ).nonzero(as_tuple=True)
    word_logits = word_logits # [task_IXS]
    word_labels = word_labels # [task_IXS]
    top1_pred = word_logits.softmax(-1).argmax(-1)
    top1_acc = (top1_pred == word_labels).float().mean()
    top5_acc = torch.isin(torch.topk(word_logits.softmax(-1), k=5, dim=-1).indices, word_labels).any(-1).float().mean()
    top1_word.append(top1_acc)
    top5_word.append(top5_acc)

n_examples = len(top1_word)

output_dict = {
    "word_top1_mean": np.stack(top1_word).mean(),
    "word_top1_sem": np.stack(top1_word).std() / np.sqrt(n_examples),
    "word_top5_mean": np.stack(top5_word).mean(),
    "word_top5_sem": np.stack(top5_word).std() / np.sqrt(n_examples),
}
output_dict

  0%|          | 0/500 [00:00<?, ?it/s]

{'word_top1_mean': np.float32(0.10292),
 'word_top1_sem': np.float64(0.0013537160945087327),
 'word_top5_mean': np.float32(0.653),
 'word_top5_sem': np.float64(0.003756221633848441)}

## Generate examples of model failure

In [ ]:
audio, labels = next(iter(test_dataloader))
word_labels = labels['signal/word_int']

In [ ]:

audio, labels = next(iter(test_dataloader))
word_labels = labels['signal/word_int']
module = module.cuda().eval()


with torch.no_grad():
    task_IXS = (word_labels != 0 ).nonzero(as_tuple=True)
    _, _ , model_preds = module(audio.cuda())
    word_preds = model_preds['signal/word_int'].cpu().softmax(-1).cpu()[task_IXS]
    model_top_1 = word_preds.argmax(-1)
    word_labels = word_labels[task_IXS] 

## Cut audio to valid ixs for display
audio = audio[task_IXS]

In [ ]:
raw_acc = (model_top_1 == word_labels).numpy().mean()
raw_acc

In [ ]:
# model top5
top_5 = torch.isin(torch.topk(word_preds, k=5, dim=-1).indices, word_labels).any(-1).float().mean()
top_5

In [ ]:
word_and_speaker_encodings = pickle.load(
    open("/mnt/home/igriffith/ceph/projects/cochdnn/robustness/audio_functions/word_and_speaker_encodings_jsinv3.pckl", "rb")
)
class_map = word_and_speaker_encodings["word_idx_to_word"]

In [ ]:
### Geck examples where model predicted wrong label 




model_failure_IXS = torch.where((model_top_1 != word_labels))[0].numpy()

for _ in range(20):

    failure_eg = int(model_failure_IXS[_])

    true_word = class_map[int(word_labels[failure_eg])]
    ## Get model top 1 and top 5 for that eg 
    model_pred = class_map[int(model_top_1[failure_eg])]

    # model top 5 transcripbed 
    model_eg_top5 = torch.topk(word_preds[failure_eg], k=5, dim=-1).indices
    model_eg_top5_words = [class_map[int(ix)] for ix in model_eg_top5] 

    print(f"True word: {true_word}")
    print(f"Model top 5 words: {', '.join(model_eg_top5_words)}")
    display(Audio(audio[failure_eg], rate=20_000, normalize=False))
    print("\n")
